In [2]:
import json
import pandas as pd
import os

data = []
base_path = os.path.dirname(os.getcwd())
log_dir = os.path.join(base_path, "log")

for file in os.listdir(log_dir):
    if file.endswith(".json"):
        file_path = os.path.join(log_dir, file)
        with open(file_path) as f:
            for line in f:
                data.append(json.loads(line))

df = pd.DataFrame(data)

print(df.head())

                  eventid          src_ip  src_port         dst_ip  dst_port  \
0  cowrie.session.connect   101.126.41.73   58982.0  172.31.45.231    2222.0   
1   cowrie.client.version   101.126.41.73       NaN            NaN       NaN   
2       cowrie.client.kex   101.126.41.73       NaN            NaN       NaN   
3   cowrie.session.closed   101.126.41.73       NaN            NaN       NaN   
4  cowrie.session.connect  175.27.145.234   18854.0  172.31.45.231    2222.0   

        session protocol                                            message  \
0  8fb6f3d375c6      ssh  New connection: 101.126.41.73:58982 (172.31.45...   
1  8fb6f3d375c6      ssh          Remote SSH version: SSH-2.0-libssh_0.11.1   
2  8fb6f3d375c6      ssh  SSH client hassh fingerprint: 03a80b21afa81068...   
3  8fb6f3d375c6      ssh                Connection lost after 120.0 seconds   
4  3921794b8300      ssh  New connection: 175.27.145.234:18854 (172.31.4...   

             sensor                         

In [ ]:
sessions = []
print(df.columns)
grouped = df.groupby("session") #Agrupas el df por sesión
for session_id, group in grouped:
    #Diccionario de esta sesion
    session_data = {}

    #Guarda ID de la sesión
    session_data["session"] = session_id

    session_data["protocol"] = group["protocol"].dropna().iloc[0] if not group["protocol"].dropna().empty else None

    # IP atacante
    session_data["src_ip"] = group["src_ip"].dropna().iloc[0] if not group["src_ip"].dropna().empty else None

    # Puerto origen 
    session_data["src_port"] = group["src_port"].dropna().iloc[0] if not group["src_port"].dropna().empty else None

    session_data["username"] = group["username"].dropna().iloc[0] if not group["username"].dropna().empty else None
    session_data["password"] = group["password"].dropna().iloc[0] if not group["password"].dropna().empty else None

    # Duración segun evento cierre
    duration = group[group["eventid"] == "cowrie.session.closed"]["duration"]
    session_data["duration"] = float(duration.iloc[0]) if not duration.empty else 0

    # Login exitoso (1 = si, 0 = no)
    session_data["login_success"] = int((group["eventid"] == "cowrie.login.success").any())

    # Intentos de login fallidos
    session_data["login_attempts"] = (group["eventid"] == "cowrie.login.failed").sum()

    # Nº comandos ejecutados por el atacante 
    session_data["num_commands"] = (group["eventid"] == "cowrie.command.input").sum()

    # Descarga de archivos en la sesión (1 = sí, 0 = no)
    session_data["file_download"] = int((group["eventid"] == "cowrie.session.file_download").any())

    #Comandos atacante
    commands = group[group["eventid"] == "cowrie.command.input"]["input"].dropna().tolist()
    session_data["commands"] = commands

    #Distintos ataques en base a comandos
    # 1. Reconocimiento
    session_data["recon"] = int(
        any(any(x in cmd.lower() for x in ["uname","whoami","id","ifconfig","pwd"]) for cmd in commands)
    )

    # 2. Descarga malware
    session_data["download"] = int(
        any(any(x in cmd.lower() for x in ["wget","curl","tftp"]) for cmd in commands)
    )

    # 3. Cambio permisos
    session_data["chmod"] = int(
        any("chmod" in cmd.lower() for cmd in commands)
    )

    # 4. Ejecución de binarios/scripts
    session_data["execution"] = int(
        any(any(x in cmd.lower() for x in ["./","sh ","bash "]) for cmd in commands)
    )

    # 5. Persistencia
    session_data["persistence"] = int(
        any(".ssh" in cmd.lower() or "authorized_keys" in cmd.lower() for cmd in commands)
    )

    #Variedad de comandos en el ataque
    session_data["unique_commands"] = len(set(commands)) 

    #Complejidad del comando
    if commands:
        avg_len = sum(map(len, commands)) / len(commands)
    else:
        avg_len = 0
    session_data["avg_command_length"] = avg_len

    # Velocidad del ataque, alto->bot, bajo -> humano
    session_data["commands_per_second"] = (
        session_data["num_commands"] / session_data["duration"]
        if session_data["duration"] > 0 else 0
    )

    # Agrega diccionario de sesión
    sessions.append(session_data)
    
# Crea dataframe a partir de todos los diccionarios
dataset = pd.DataFrame(sessions)

print(dataset.head(50))

Index(['eventid', 'src_ip', 'src_port', 'dst_ip', 'dst_port', 'session',
       'protocol', 'message', 'sensor', 'uuid', 'timestamp', 'version',
       'hassh', 'hasshAlgorithms', 'kexAlgs', 'keyAlgs', 'encCS', 'macCS',
       'compCS', 'langCS', 'duration', 'username', 'password', 'arch', 'input',
       'ttylog', 'size', 'shasum', 'duplicate', 'outfile', 'destfile',
       'command', 'option_name', 'option_byte', 'width', 'height'],
      dtype='str')
         session           src_ip  src_port        username  \
0   0033683edcd9      65.20.175.6   35567.0          centos   
1   00ae7ffbb20a    117.216.33.31   40544.0           admin   
2   00dd83768373  220.246.194.124   47791.0           admin   
3   010705ba79a9   35.130.111.146   35155.0           admin   
4   0175c5fd6f01     183.82.97.80   39308.0          config   
5   01efdd37d845      177.174.0.3   56125.0            User   
6   020c2cb3291e    120.194.50.39   12292.0           blank   
7   0264028b595e   162.216.149.27   62

In [4]:
# Cuenta de cuantas veces nos ha atacado cada IP.
group_by_ip = dataset["src_ip"].value_counts()
print(group_by_ip)


src_ip
101.126.41.73      35
45.194.37.246      28
102.210.149.130    21
36.94.137.107      11
183.247.171.186     7
                   ..
58.115.51.223       1
183.171.56.123      1
85.229.46.153       1
102.90.34.90        1
180.76.104.208      1
Name: count, Length: 905, dtype: int64


In [5]:
import geoip2.database

# Abrir la base de datos local
dic = {}
reader = geoip2.database.Reader('../db/GeoLite2-City.mmdb')
for i,k in  group_by_ip.items():

    response = reader.city(i)
    if response.country.name not in dic.keys():
        dic[response.country.name] = k
    else:
        dic[response.country.name] += k

reader.close()

print(dic)


ModuleNotFoundError: No module named 'geoip2'

In [ ]:
# Comandos ejecutados

dataset_filtrado = dataset[dataset["commands"].str.len() > 0]
print(dataset_filtrado["commands"])

36                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

In [ ]:
low = 0
med = 0
high = 0
for i in dataset["duration"]:
    if i < 20:
        low += 1
    if 20 <= i < 80:
        med += 1
    if i >= 80:
        high += 1
print("Low duration connections: ", low)
print("Medium duration connections: ", med)
print("High duration connections: ", high)



Low duration connections:  631
Medium duration connections:  23
High duration connections:  121


In [ ]:
print(dataset["protocol"].value_counts())